In [ ]:
# ========================================
# PLATFORM SETUP — Choose ONE option
# ========================================

# --- OPTION A: KAGGLE (default) ---
import os, sys
KAGGLE_DATASET = '/kaggle/input/datasets/harshraj9988/brain-tumor-project-files'  # <-- adjust if needed
sys.path.append(KAGGLE_DATASET)
IMAGES_DIR = os.path.join(KAGGLE_DATASET, 'data', 'segmentation', 'images')
MASKS_DIR = os.path.join(KAGGLE_DATASET, 'data', 'segmentation', 'masks')
MODEL_DIR = '/kaggle/working/models'
RESULTS_DIR = '/kaggle/working/results'
BASE_DIR = '/kaggle/working'

# --- OPTION B: GOOGLE COLAB (uncomment below, comment out Option A) ---
# from google.colab import drive
# drive.mount('/content/drive')
# import os, sys
# os.chdir('/content/drive/MyDrive/brain-tumor-project/notebooks')
# BASE_DIR = os.path.dirname(os.getcwd())
# sys.path.append(BASE_DIR)
# IMAGES_DIR = os.path.join(BASE_DIR, 'data', 'segmentation', 'images')
# MASKS_DIR = os.path.join(BASE_DIR, 'data', 'segmentation', 'masks')
# MODEL_DIR = os.path.join(BASE_DIR, 'models')
# RESULTS_DIR = os.path.join(BASE_DIR, 'results')

os.makedirs(MODEL_DIR, exist_ok=True)
os.makedirs(RESULTS_DIR, exist_ok=True)
print('Setup complete!')


In [ ]:
import os
os.environ['SM_FRAMEWORK'] = 'tf.keras'
import segmentation_models as sm
print(f'segmentation_models loaded: {sm.__version__}')

# 13 - Brain Tumor Segmentation with Attention U-Net + EfficientNet Encoder

**Architecture:** U-Net with **pretrained EfficientNetB3** encoder + **scSE Attention** decoder

**Attention Mechanism:** concurrent Spatial and Channel Squeeze & Excitation (scSE)
adds attention at each decoder block — learns WHICH channels and WHERE spatially to focus.

**Why pretrained?** Our dataset is only 3,064 images. Pretrained ImageNet weights give the
encoder a massive head start — it already knows edges, textures, and shapes.

**Dataset:** 3,064 grayscale MRI + binary masks | **Loss:** Dice + Focal


In [ ]:
import os, sys
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
from tqdm import tqdm
import warnings; warnings.filterwarnings('ignore')

import tensorflow as tf
from tensorflow.keras import backend as K
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping, ReduceLROnPlateau
from sklearn.model_selection import train_test_split

print(f'TensorFlow: {tf.__version__}')
print(f'GPU: {tf.config.list_physical_devices("GPU")}')

IMG_SIZE = 256
BATCH_SIZE = 16
EPOCHS = 80

## 1. Load Segmentation Data

In [ ]:
images = []
masks = []

image_files = sorted(os.listdir(IMAGES_DIR), key=lambda x: int(os.path.splitext(x)[0]))
mask_files = sorted(os.listdir(MASKS_DIR), key=lambda x: int(os.path.splitext(x)[0]))
print(f'Found {len(image_files)} images, {len(mask_files)} masks')

for img_f, mask_f in tqdm(zip(image_files, mask_files), total=len(image_files), desc='Loading'):
    assert os.path.splitext(img_f)[0] == os.path.splitext(mask_f)[0], f'Mismatch: {img_f} vs {mask_f}'
    
    # Load grayscale image → stack to 3 channels (pretrained encoder expects 3ch)
    img = Image.open(os.path.join(IMAGES_DIR, img_f)).convert('L')
    img = img.resize((IMG_SIZE, IMG_SIZE), Image.LANCZOS)
    img_arr = np.array(img, dtype=np.float32) / 255.0
    img_arr = np.stack([img_arr]*3, axis=-1)
    images.append(img_arr)
    
    # Load binary mask
    mask = Image.open(os.path.join(MASKS_DIR, mask_f)).convert('L')
    mask = mask.resize((IMG_SIZE, IMG_SIZE), Image.NEAREST)
    mask_arr = (np.array(mask, dtype=np.float32) > 127).astype(np.float32)
    masks.append(mask_arr)

X = np.array(images, dtype=np.float32)
y = np.expand_dims(np.array(masks, dtype=np.float32), axis=-1)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f'Train: {X_train.shape} | Test: {X_test.shape}')
print(f'Tumor pixel ratio: {y_train.mean()*100:.2f}%')


## 2. Visualize Samples

In [ ]:
fig, axes = plt.subplots(3, 4, figsize=(16, 12))
for i in range(3):
    idx = np.random.randint(0, len(X_train))
    axes[i,0].imshow(X_train[idx,:,:,0], cmap='gray'); axes[i,0].set_title('MRI', fontweight='bold'); axes[i,0].axis('off')
    axes[i,1].imshow(y_train[idx].squeeze(), cmap='gray'); axes[i,1].set_title('Mask', fontweight='bold'); axes[i,1].axis('off')
    axes[i,2].imshow(X_train[idx,:,:,0], cmap='gray'); axes[i,2].imshow(y_train[idx].squeeze(), cmap='Reds', alpha=0.5)
    axes[i,2].set_title('Overlay', fontweight='bold'); axes[i,2].axis('off')
    area = y_train[idx].mean()*100
    axes[i,3].text(0.5,0.5,f'Tumor: {area:.1f}%',transform=axes[i,3].transAxes,fontsize=16,ha='center',va='center',fontweight='bold'); axes[i,3].axis('off')
plt.suptitle('Segmentation Data Samples', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(BASE_DIR, 'segmentation_samples.png'), dpi=150, bbox_inches='tight')
plt.show()


## 3. Data Augmentation

In [ ]:
def augment(image, mask):
    if tf.random.uniform(()) > 0.5:
        image = tf.image.flip_left_right(image)
        mask = tf.image.flip_left_right(mask)
    if tf.random.uniform(()) > 0.5:
        image = tf.image.flip_up_down(image)
        mask = tf.image.flip_up_down(mask)
    k = tf.random.uniform((), 0, 4, dtype=tf.int32)
    image = tf.image.rot90(image, k)
    mask = tf.image.rot90(mask, k)
    image = tf.image.random_brightness(image, 0.1)
    image = tf.image.random_contrast(image, 0.9, 1.1)
    image = tf.clip_by_value(image, 0.0, 1.0)
    return image, mask

train_ds = tf.data.Dataset.from_tensor_slices((X_train, y_train)).shuffle(2000)
train_ds = train_ds.map(augment, num_parallel_calls=tf.data.AUTOTUNE).batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)
val_ds = tf.data.Dataset.from_tensor_slices((X_test, y_test)).batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)
print(f'Train batches: {len(train_ds)} | Val batches: {len(val_ds)}')


## 4. Build Attention U-Net with Pretrained EfficientNetB3 Backbone

Uses `segmentation_models` library with `decoder_attention_type='scse'`.

**scSE = concurrent Spatial and Channel Squeeze & Excitation**
- Channel SE: learns which feature channels are important for tumor detection
- Spatial SE: learns which spatial locations contain tumor-relevant information

This is a proven, production-grade attention mechanism for medical image segmentation.

In [ ]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

In [ ]:
# ============================================================================
# SQUEEZE-AND-EXCITATION (SE) ATTENTION BLOCK
# ============================================================================
class SEAttentionBlock(layers.Layer):
    """Squeeze-and-Excitation attention for channel-wise feature recalibration"""
    
    def __init__(self, reduction_ratio=16, **kwargs):
        super(SEAttentionBlock, self).__init__(**kwargs)
        self.reduction_ratio = reduction_ratio
    
    def build(self, input_shape):
        channels = input_shape[-1]
        self.fc1 = layers.Dense(max(channels // self.reduction_ratio, 1), activation='relu')
        self.fc2 = layers.Dense(channels, activation='sigmoid')
        super().build(input_shape)
    
    def call(self, x):
        squeeze = layers.GlobalAveragePooling2D(keepdims=True)(x)
        excitation = self.fc1(squeeze)
        excitation = self.fc2(excitation)
        return x * excitation


class SpatialAttentionBlock(layers.Layer):
    """Spatial attention focusing on where to attend"""
    
    def __init__(self, kernel_size=7, **kwargs):
        super(SpatialAttentionBlock, self).__init__(**kwargs)
        self.kernel_size = kernel_size
    
    def build(self, input_shape):
        self.conv = layers.Conv2D(
            filters=1,
            kernel_size=self.kernel_size,
            padding='same',
            activation='sigmoid',
            use_bias=False
        )
        super().build(input_shape)
    
    def call(self, x):
        avg_pool = layers.Lambda(lambda x: tf.reduce_mean(x, axis=-1, keepdims=True))(x)
        max_pool = layers.Lambda(lambda x: tf.reduce_max(x, axis=-1, keepdims=True))(x)
        concat = layers.Concatenate(axis=-1)([avg_pool, max_pool])
        attention_map = self.conv(concat)
        return x * attention_map


class scSEAttentionBlock(layers.Layer):
    """Combined Spatial and Channel Squeeze-Excitation attention"""
    
    def __init__(self, reduction_ratio=16, kernel_size=7, **kwargs):
        super(scSEAttentionBlock, self).__init__(**kwargs)
        self.se_attention = SEAttentionBlock(reduction_ratio=reduction_ratio)
        self.spatial_attention = SpatialAttentionBlock(kernel_size=kernel_size)
    
    def call(self, x):
        x = self.se_attention(x)
        x = self.spatial_attention(x)
        return x

In [ ]:
BACKBONE = 'resnet50'
base_model = sm.Unet(
    BACKBONE,
    input_shape=(IMG_SIZE, IMG_SIZE, 3),
    classes=1,
    activation='sigmoid',
    encoder_weights='imagenet',
    encoder_freeze=False,
)

inputs = keras.Input(shape=(IMG_SIZE, IMG_SIZE, 3))
x = base_model(inputs)
x = scSEAttentionBlock(reduction_ratio=16, kernel_size=7)(x)
model = keras.Model(inputs=inputs, outputs=x)

print(f'Model: Attention U-Net (scSE) + {BACKBONE}')
print(f'Parameters: {model.count_params():,}')
trainable_params = sum([tf.keras.backend.count_params(w) for w in model.trainable_weights])
print(f'Trainable: {trainable_params:,}')

## 5. Loss: Dice + Focal (from segmentation_models)

In [ ]:
# segmentation_models provides optimized loss functions
dice_loss = sm.losses.DiceLoss()
focal_loss = sm.losses.BinaryFocalLoss(alpha=0.75, gamma=2.0)
total_loss = dice_loss + focal_loss  # Combined!

# Metrics
iou_score = sm.metrics.IOUScore(threshold=0.5)
f1_score_metric = sm.metrics.FScore(threshold=0.5)

# Custom dice for display
def dice_coefficient(y_true, y_pred, smooth=1.0):
    y_true_f = K.flatten(y_true)
    y_pred_f = K.flatten(y_pred)
    intersection = K.sum(y_true_f * y_pred_f)
    return (2.*intersection+smooth) / (K.sum(y_true_f)+K.sum(y_pred_f)+smooth)

print('Loss: Dice Loss + Binary Focal Loss (alpha=0.75, gamma=2.0)')
print('Focal loss handles the extreme class imbalance (tumors are < 1% of pixels)')


## 6. Training

In [ ]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, backend as K

class SEAttentionBlock(layers.Layer):
    def __init__(self, reduction_ratio=16, **kwargs):
        super(SEAttentionBlock, self).__init__(**kwargs)
        self.reduction_ratio = reduction_ratio
    def build(self, input_shape):
        channels = input_shape[-1]
        self.fc1 = layers.Dense(max(channels // self.reduction_ratio, 1), activation='relu')
        self.fc2 = layers.Dense(channels, activation='sigmoid')
        super().build(input_shape)
    def call(self, x):
        squeeze = layers.GlobalAveragePooling2D(keepdims=True)(x)
        excitation = self.fc1(squeeze)
        excitation = self.fc2(excitation)
        return x * excitation

class SpatialAttentionBlock(layers.Layer):
    def __init__(self, kernel_size=7, **kwargs):
        super(SpatialAttentionBlock, self).__init__(**kwargs)
        self.kernel_size = kernel_size
    def build(self, input_shape):
        self.conv = layers.Conv2D(filters=1, kernel_size=self.kernel_size, padding='same', activation='sigmoid', use_bias=False)
        super().build(input_shape)
    def call(self, x):
        avg_pool = layers.Lambda(lambda x: tf.reduce_mean(x, axis=-1, keepdims=True))(x)
        max_pool = layers.Lambda(lambda x: tf.reduce_max(x, axis=-1, keepdims=True))(x)
        concat = layers.Concatenate(axis=-1)([avg_pool, max_pool])
        attention_map = self.conv(concat)
        return x * attention_map

class scSEAttentionBlock(layers.Layer):
    def __init__(self, reduction_ratio=16, kernel_size=7, **kwargs):
        super(scSEAttentionBlock, self).__init__(**kwargs)
        self.se_attention = SEAttentionBlock(reduction_ratio=reduction_ratio)
        self.spatial_attention = SpatialAttentionBlock(kernel_size=kernel_size)
    def call(self, x):
        x = self.se_attention(x)
        x = self.spatial_attention(x)
        return x

def focal_loss(y_true, y_pred, alpha=0.75, gamma=2.0):
    epsilon = 1e-7
    y_pred = K.clip(y_pred, epsilon, 1. - epsilon)
    ce = -y_true * K.log(y_pred) - (1 - y_true) * K.log(1 - y_pred)
    focal_weight = K.pow(1 - y_pred, gamma)
    focal_loss_value = alpha * focal_weight * ce
    return K.mean(focal_loss_value)

def dice_loss(y_true, y_pred):
    smooth = 1e-5
    y_true_f = K.flatten(y_true)
    y_pred_f = K.flatten(y_pred)
    intersection = K.sum(y_true_f * y_pred_f)
    union = K.sum(y_true_f) + K.sum(y_pred_f)
    dice = (2.0 * intersection + smooth) / (union + smooth)
    return 1 - dice

def combined_loss(y_true, y_pred):
    focal = focal_loss(y_true, y_pred)
    dice = dice_loss(y_true, y_pred)
    return 0.4 * focal + 0.6 * dice

In [ ]:
# Compile with better loss function for imbalanced data
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=5e-5),
    loss=combined_loss,
    metrics=[dice_coefficient, iou_score, f1_score_metric, 'accuracy']
)

callbacks = [
    ModelCheckpoint(
        filepath=os.path.join(MODEL_DIR, 'attention_unet_best.h5'),
        monitor='val_dice_coefficient', save_best_only=True, mode='max', verbose=1),
    EarlyStopping(
        monitor='val_dice_coefficient', patience=20, restore_best_weights=True, mode='max', verbose=1),
    ReduceLROnPlateau(
        monitor='val_loss', factor=0.5, patience=7, min_lr=1e-7, verbose=1)
]

import time; start_time = time.time()
history = model.fit(train_ds, epochs=EPOCHS, validation_data=val_ds, callbacks=callbacks, verbose=1)
training_time = time.time() - start_time
print(f'\nDone in {training_time/60:.1f} min')


## 7. Training History

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
axes[0].plot(history.history['dice_coefficient'], label='Train', color='#FF6B6B', lw=2)
axes[0].plot(history.history['val_dice_coefficient'], label='Val', color='#4ECDC4', lw=2)
axes[0].set_title('Dice Coefficient', fontsize=14, fontweight='bold')
axes[0].legend(); axes[0].grid(True, alpha=0.3)
axes[1].plot(history.history['iou_score'], label='Train', color='#FF6B6B', lw=2)
axes[1].plot(history.history['val_iou_score'], label='Val', color='#4ECDC4', lw=2)
axes[1].set_title('IoU Score', fontsize=14, fontweight='bold')
axes[1].legend(); axes[1].grid(True, alpha=0.3)
axes[2].plot(history.history['loss'], label='Train', color='#FF6B6B', lw=2)
axes[2].plot(history.history['val_loss'], label='Val', color='#4ECDC4', lw=2)
axes[2].set_title('Loss', fontsize=14, fontweight='bold')
axes[2].legend(); axes[2].grid(True, alpha=0.3)
plt.suptitle('Attention U-Net (scSE) + EfficientNetB3 Training', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(BASE_DIR, 'attention_unet_training_history.png'), dpi=150, bbox_inches='tight')
plt.show()


## 8. Evaluation

In [ ]:
custom_objects = {
    'combined_loss': combined_loss,
    'dice_coefficient': dice_coefficient,
    'iou_score': iou_score,
    'f1_score_metric': f1_score_metric,
    'scSEAttentionBlock': scSEAttentionBlock,
}

with tf.keras.utils.custom_object_scope(custom_objects):
    model = tf.keras.models.load_model(best_path)
    print('Loaded best model')

# Recompile with your custom metrics
model.compile(
    optimizer='adam',                  # use whatever optimizer you trained with
    loss=combined_loss,
    metrics=[dice_coefficient, iou_score, f1_score_metric, 'accuracy']
)

test_results = model.evaluate(X_test, y_test, verbose=1)
print(f'\n{"="*50}')
print('ATTENTION U-NET (scSE) + ResNet50 - TEST RESULTS')
print(f'{"="*50}')
print(f'Dice Coefficient: {test_results[1]:.4f}')
print(f'IoU Score:        {test_results[2]:.4f}')
print(f'F1 Score:         {test_results[3]:.4f}')
print(f'Accuracy:         {test_results[4]:.4f}')

## 9. Prediction Visualization

In [ ]:
fig, axes = plt.subplots(5, 4, figsize=(16, 20))
indices = np.random.choice(len(X_test), 5, replace=False)
for row, idx in enumerate(indices):
    d = dice_scores[idx]
    axes[row,0].imshow(X_test[idx,:,:,0], cmap='gray'); axes[row,0].set_title('MRI', fontweight='bold'); axes[row,0].axis('off')
    axes[row,1].imshow(y_test[idx].squeeze(), cmap='gray'); axes[row,1].set_title('Ground Truth', fontweight='bold'); axes[row,1].axis('off')
    axes[row,2].imshow(y_pred_bin[idx].squeeze(), cmap='gray'); axes[row,2].set_title(f'Predicted (Dice:{d:.3f})', fontweight='bold'); axes[row,2].axis('off')
    axes[row,3].imshow(X_test[idx,:,:,0], cmap='gray'); axes[row,3].imshow(y_pred_bin[idx].squeeze(), cmap='Reds', alpha=0.4)
    axes[row,3].set_title('Overlay', fontweight='bold'); axes[row,3].axis('off')
plt.suptitle('Attention U-Net (scSE) + EfficientNetB3 Predictions', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(BASE_DIR, 'attention_unet_predictions.png'), dpi=150, bbox_inches='tight')
plt.show()


## 10. Dice Score Distribution

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
ax.hist(dice_scores, bins=50, color='#4ECDC4', edgecolor='white', alpha=0.8)
ax.axvline(np.mean(dice_scores), color='red', ls='--', lw=2, label=f'Mean: {np.mean(dice_scores):.4f}')
ax.set_title('Per-Sample Dice Distribution', fontsize=14, fontweight='bold')
ax.set_xlabel('Dice'); ax.set_ylabel('Count'); ax.legend(fontsize=12); ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(BASE_DIR, 'attention_unet_dice_distribution.png'), dpi=150, bbox_inches='tight')
plt.show()


## 11. Save Results

In [ ]:
total_params = model.count_params()
trainable_params = sum([tf.size(w).numpy() for w in model.trainable_weights])

print(f'Total params:     {total_params:,}')
print(f'Trainable params: {trainable_params:,}')

In [ ]:
import json as json_lib
seg_results = {
    'model': f'Attention U-Net (scSE) + {BACKBONE}',
    'input_size': IMG_SIZE,
    'total_params': int(total_params),
    'trainable_params': int(trainable_params),
    'test_dice': round(float(test_results[1]), 4),
    'test_iou': round(float(test_results[2]), 4),
    'test_f1': round(float(test_results[3]), 4),
    'test_accuracy': round(float(test_results[4]), 4),
    'mean_sample_dice': round(float(np.mean(dice_scores)), 4),
    'epochs_trained': len(history.history['loss']),
    'training_time_sec': round(training_time, 1),
}
with open(os.path.join(RESULTS_DIR, 'attention_unet_results.json'), 'w') as f:
    json_lib.dump(seg_results, f, indent=2)
print('Results saved!')
print(f'Final Dice: {seg_results["test_dice"]}')
print(f'Final IoU:  {seg_results["test_iou"]}')
